# NumberOFF ナンバー検出モデルの追加学習

上から順に ▶ を押していくだけで進むノートです。途中で人の作業（ナンバーに枠を付ける）が1回だけ入ります。

**最初にやること**
1. メニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選んで保存
2. Googleドライブに `NumberOFF/raw` フォルダを作り、学習に使う写真を入れる
   - 目安：300〜500枚（最低でも150枚）
   - 夜・斜め・何台も並んだ写真・遠くの車を多めに
   - スマホの縦長写真や、ナンバーが写真の端で切れかけている写真も入れる
   - **ナンバーが写っていない写真やヘッドライト・看板が目立つ写真も50枚くらい**混ぜる（誤検出が減る）
   - 自分の写真か、写っている人の了解をもらった写真だけにする

フォルダの中身は最終的にこうなります：
```
NumberOFF/
  raw/            ← 元の写真（あなたが入れる）
  images/         ← 向きと大きさをそろえた写真（ステップ1で自動作成）
  prelabels.zip   ← 今のモデルが付けた下書きの枠（ステップ1で自動作成）
  images.zip      ← 枠付け作業用にまとめた写真（ステップ1で自動作成）
  labels.zip      ← 枠を直したもの（あなたがアップロード）
  split.json      ← 学習用・確認用・テスト用の振り分け（固定。消さない）
  runs/           ← 学習の記録
  model.onnx      ← 完成したモデル（ステップ6で自動作成）
```

## 0. 準備（毎回最初に実行）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install "ultralytics>=8.3,<8.4" onnx onnxruntime huggingface_hub pillow-heif
import os, json, glob, shutil, zipfile, hashlib, math, random
from pathlib import Path
from PIL import Image, ImageOps   # 途中のステップから実行し直しても動くように、ここで読み込む
ROOT = Path('/content/drive/MyDrive/NumberOFF')
for d in ['raw','images','runs']: (ROOT/d).mkdir(parents=True, exist_ok=True)
print('写真の枚数(raw):', len([p for p in (ROOT/'raw').rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png','.heic','.webp')]))

## 0.5 元になる今のモデルを取得
今アプリで使っているモデル（YOLO11n・ナンバー検出用）の学習用ファイルを取ってきます。これに日本のナンバーを追加で覚えさせます。

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
REPO = 'morsetechlab/yolov11-license-plate-detection'
files = list_repo_files(REPO)
pts = [f for f in files if f.endswith('.pt')]
print('見つかったファイル:', pts)
cand = [f for f in pts if 'v1n' in f] or [f for f in pts if 'n.pt' in f] or pts
if cand:
    BASE = hf_hub_download(REPO, cand[0], local_dir='/content/base')
else:
    BASE = 'yolo11n.pt'   # 見つからないときは汎用のYOLO11nから学習（写真は多めに）
    print('⚠ ナンバー用の元モデルが見つからなかったので、汎用モデルから学習します')
print('元モデル:', BASE)

## 1. 写真をそろえて、今のモデルで下書きの枠を付ける
- 写真の向き（スマホの縦横）を直して、長辺2560pxにそろえます
- 今のモデルでナンバーを探して、下書きの枠を作ります（これを直すだけなので枠付けが速くなります）

In [ ]:
from PIL import Image, ImageOps
import pillow_heif; pillow_heif.register_heif_opener()   # iPhoneのHEIC写真も読めるように
IMG = ROOT/'images'
n=0
for p in sorted((ROOT/'raw').rglob('*')):
    if p.suffix.lower() not in ('.jpg','.jpeg','.png','.heic','.webp'): continue
    out = IMG/(p.stem+'.jpg')
    if out.exists(): continue
    im = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
    im.thumbnail((2560,2560), Image.LANCZOS)
    im.save(out, quality=92)   # EXIF（位置情報など）は付けずに保存
    n+=1
print('そろえた写真:', n, '／ 合計', len(list(IMG.glob('*.jpg'))))

from ultralytics import YOLO
m = YOLO(BASE)
PRE = Path('/content/prelabels'); shutil.rmtree(PRE, ignore_errors=True); PRE.mkdir()
for r in m.predict(source=str(IMG), imgsz=1280, conf=0.2, stream=True, verbose=False):
    lines=[f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for x,y,w,h in r.boxes.xywhn.tolist()]
    (PRE/(Path(r.path).stem+'.txt')).write_text('\n'.join(lines))
(PRE/'labels.txt').write_text('plate\n')
shutil.make_archive(str(ROOT/'prelabels'), 'zip', PRE)
shutil.make_archive(str(ROOT/'images'), 'zip', IMG)
print('ドライブに images.zip と prelabels.zip を作りました')

## 2. 枠を直す（人の作業）

ブラウザだけで動く無料ツール **makesense.ai** を使います。写真はどこにもアップロードされません（パソコンのブラウザ内だけで処理）。

1. ドライブから `NumberOFF/images.zip` と `NumberOFF/prelabels.zip` をパソコンにダウンロードして、両方とも解凍する
2. https://www.makesense.ai を開いて「Get Started」→ 解凍した**写真を全部**ドラッグ →「Object Detection」
3. ラベル名を聞かれたら `plate` を1つだけ作る →「Start project」
4. 上のメニュー「Actions」→「Import Annotations」→「Multiple files」で **prelabels の中の .txt と labels.txt を全部**選んで読み込む（YOLO形式）
5. 1枚ずつ見て直す
   - 枠がずれていたら角をドラッグ、余計な枠は選んで Delete、足りなければドラッグで追加
   - **ナンバーの白い板（黄色・緑の板）の外周ぴったり**に合わせる
   - 斜めのナンバーは、板全体が入る最小の四角で囲む
   - ナンバーがない写真は枠ゼロのままでOK
6. 「Actions」→「Export Annotations」→「A .zip package containing files in YOLO format」
7. 出てきた zip の名前を `labels.zip` にして、ドライブの `NumberOFF/` に入れる

目安：1枚5〜15秒。400枚で1時間前後です。

## 3. 学習用・確認用・テスト用に分けて、学習データを作る
テスト用（15%）は毎回同じ写真になるよう固定します。次に写真を足したときも、前のモデルとの比較がずっとできます。

In [ ]:
LBL = Path('/content/labels'); shutil.rmtree(LBL, ignore_errors=True); LBL.mkdir()
with zipfile.ZipFile(ROOT/'labels.zip') as z: z.extractall(LBL)
txts = {p.stem:p for p in LBL.rglob('*.txt') if p.name!='labels.txt'}
imgs = sorted((ROOT/'images').glob('*.jpg'))
missing = [p.stem for p in imgs if p.stem not in txts]
print('写真', len(imgs), '／ 枠ファイル', len(txts), '／ 枠なし（ナンバーなしとして扱う）', len(missing))
if len(missing) > len(imgs)*0.3: print('⚠ 枠のない写真が多いです。makesense のエクスポート漏れがないか確認してください')

SPLIT = ROOT/'split.json'
split = json.loads(SPLIT.read_text()) if SPLIT.exists() else {}
for p in imgs:
    if p.stem in split: continue
    h = int(hashlib.md5(p.stem.encode()).hexdigest(),16)%100
    split[p.stem] = 'test' if h<15 else 'val' if h<30 else 'train'
SPLIT.write_text(json.dumps(split, ensure_ascii=False, indent=1))

def read_boxes(stem):
    p = txts.get(stem)
    if not p: return []
    out=[]
    for ln in p.read_text().split('\n'):
        v=ln.split()
        if len(v)==5: out.append([float(a) for a in v[1:]])
    return out

# アプリと同じように写真を区切って拡大した切り抜きも学習に入れる（遠くの小さいナンバーに強くなる）
def tiles(W,H):
    T=max(1000,math.sqrt(W*H/12)); nx=math.ceil(W/T); ny=math.ceil(H/T)
    if nx*ny<=1: return []
    tw=min(W,math.ceil(W/nx*1.25)); th=min(H,math.ceil(H/ny*1.25))
    return [(0 if nx==1 else round(i*(W-tw)/(nx-1)), 0 if ny==1 else round(j*(H-th)/(ny-1)), tw, th) for j in range(ny) for i in range(nx)]

DS = Path('/content/ds'); shutil.rmtree(DS, ignore_errors=True)
for s in ['train','val','test']: (DS/'images'/s).mkdir(parents=True); (DS/'labels'/s).mkdir(parents=True)
cnt={'train':0,'val':0,'test':0,'tiles':0}
for p in imgs:
    s=split[p.stem]; bx=read_boxes(p.stem)
    shutil.copy(p, DS/'images'/s/p.name)
    (DS/'labels'/s/(p.stem+'.txt')).write_text('\n'.join(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for x,y,w,h in bx)); cnt[s]+=1
    if s!='train': continue
    im=Image.open(p); W,H=im.size
    for k,(tx,ty,tw,th) in enumerate(tiles(W,H)):
        keep=[]; cut=False
        for x,y,w,h in bx:
            x0,y0,x1,y1=(x-w/2)*W,(y-h/2)*H,(x+w/2)*W,(y+h/2)*H
            ix0,iy0,ix1,iy1=max(x0,tx),max(y0,ty),min(x1,tx+tw),min(y1,ty+th)
            if ix1<=ix0 or iy1<=iy0: continue
            if (ix1-ix0)*(iy1-iy0) < 0.6*(x1-x0)*(y1-y0): cut=True; break   # ナンバーが途中で切れる切り抜きは使わない
            keep.append((((ix0+ix1)/2-tx)/tw, ((iy0+iy1)/2-ty)/th, (ix1-ix0)/tw, (iy1-iy0)/th))
        if cut or (not keep and random.Random(p.stem+str(k)).random()>0.25): continue
        name=f"{p.stem}__t{k}"
        im.crop((tx,ty,tx+tw,ty+th)).save(DS/'images'/'train'/(name+'.jpg'), quality=92)
        (DS/'labels'/'train'/(name+'.txt')).write_text('\n'.join(f"0 {a:.6f} {b:.6f} {c:.6f} {d:.6f}" for a,b,c,d in keep)); cnt['tiles']+=1
(DS/'data.yaml').write_text(f"path: {DS}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: plate\n")
print(cnt)

## 4. 学習する（T4 GPUで30〜60分）
途中でブラウザを閉じると止まります。学習の記録はドライブの `NumberOFF/runs` に残ります。

In [ ]:
from ultralytics import YOLO
model = YOLO(BASE)
model.train(data=str(DS/'data.yaml'), imgsz=640, epochs=80, patience=20, batch=16,
            lr0=0.002, lrf=0.1, warmup_epochs=2, close_mosaic=10,
            degrees=8, shear=4, perspective=0.0005, hsv_v=0.5, fliplr=0.0,   # 斜め・暗さに強く。文字が逆になる左右反転はしない
            project=str(ROOT/'runs'), name='jp', exist_ok=True, plots=True)
NEW = ROOT/'runs'/'jp'/'weights'/'best.pt'
print('完成:', NEW)

## 5. 今のモデルと比べる（テスト用の写真で）
- **見逃しの少なさ（Recall）**：本物のナンバーのうち見つけられた割合。高いほど見逃しが少ない
- **誤検出の少なさ（Precision）**：見つけたもののうち本当にナンバーだった割合。高いほど誤検出が少ない
- **mAP50**：総合点

新しいモデルが両方とも上回っていれば入れ替えOKです。

In [ ]:
import pandas as pd
rows=[]
for name,w in [('今のモデル',BASE),('新しいモデル',str(NEW))]:
    for sz in [640,1280]:
        r=YOLO(w).val(data=str(DS/'data.yaml'), split='test', imgsz=sz, conf=0.25, verbose=False, plots=False)
        rows.append({'モデル':name,'入力サイズ':sz,'見逃しの少なさ(Recall)':round(r.box.mr,3),'誤検出の少なさ(Precision)':round(r.box.mp,3),'総合(mAP50)':round(r.box.map50,3)})
pd.DataFrame(rows)

## 6. アプリ用に書き出す
アプリと同じ形（入力 640×640、出力 1×5×8400）の `model.onnx` を作って、ドライブに保存します。

In [ ]:
out = YOLO(str(NEW)).export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
import onnxruntime as ort, numpy as np
s = ort.InferenceSession(out, providers=['CPUExecutionProvider'])
y = s.run(None, {s.get_inputs()[0].name: np.zeros((1,3,640,640),np.float32)})[0]
print('入力', s.get_inputs()[0].shape, '出力', y.shape)
assert tuple(y.shape)==(1,5,8400), '形がアプリと合いません'
shutil.copy(out, ROOT/'model.onnx')
print('ドライブに model.onnx を保存しました（%.1f MB）' % (os.path.getsize(ROOT/'model.onnx')/1048576))

## 7. アプリに入れる

1. ドライブの `NumberOFF/model.onnx` をダウンロードして Claude に渡す
2. Claude が確認用の URL（プレビュー）でモデルを入れ替える。いつもの写真（夜・3台・縦長など）で、今のモデルより見逃しと誤検出が減っているか確かめる
3. 良ければ本番に入れる（Service Worker が新しいモデルを自動で配り直す）

**次の回**：写真を `raw` に足して、ステップ1から同じ手順。`split.json` は消さないでください（テスト用の写真が固定され、前のモデルとずっと比べられます）。
